In [22]:
include("../Envs/Env.jl")
include("../Algorithms/PPO-RNN.jl")

evaluate (generic function with 1 method)

## 1. Prepare Environment

In [23]:
pomdp = LightDark1D()
pomdp_name = "LightDark"
bool_full_observability = false
env = Env(pomdp, bool_full_observability)
action_space = GetActionSpace(env)
function create_env()
    return Env(pomdp, bool_full_observability)
end

# define convert_o function
# function POMDPs.convert_o(T::Type{<:AbstractArray}, o::Int64, m::LightDark1D)
#     vec = zeros(Float32, 3)
#     vec[o] = 1.0f0
#     return vec
# end

# define process action function
function process_action(action_index::Int, action_space::UnitRange{Int})
    action = action_space[action_index]
    len = length(action_space)
    idx = action - first(action_space) + 1
    (idx < 1 || idx > len) && error("Action $action not in action space")
    onehot = zeros(Float32, len)
    onehot[idx] = 1.0f0
    return onehot
end

process_action (generic function with 1 method)

## 2. Prepare Parameters

In [24]:
state_dim = GetObsDim(env)
layer_size = 16
rnn_hidden_size = 32
gamma = discount(pomdp)
training_episodes = 10000
batch_size = 1024

LightDark1DState(-1, 4.050878436666986)
Float32[3.7157013]


1024

## 3. Prepare PPO-RNN agent

In [25]:
# if want to use gpu, need to uncomment the below line, and use device=Flux.gpu
# using CUDA

agent = PPORNNAgent(action_space, state_dim;
    hidden_dim=layer_size, 
    rnn_hidden_size=rnn_hidden_size, 
    batch_size=batch_size, 
    device=Flux.cpu) 

PPORNNAgent(Chain(LSTM(4 => 32), Dense(32 => 16, tanh), Dense(16 => 3)), Chain(Dense(4 => 16, tanh), LSTM(16 => 32), Dense(32 => 16, tanh), Dense(16 => 1)), (layers = ((cell = (Wi = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0; … ; 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0], Float32[0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0; … ; 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0], (0.9, 0.999))), Wh = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), bias = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], (0.9, 0.999)))),), (weight = Leaf

## 4. Train

In [ ]:
# 训练
rewards, losses, evals = train!(create_env, agent, training_episodes)

Progress:   1%|█                                        |  ETA: 12:55:5526m

Update 100 | Threads: 16 | Reward: -7.38 | Policy Loss: -0.0215 | Value Loss: 0.9881 | Eval: 0.0


Progress:   1%|█                                        |  ETA: 12:42:52

## 5. Evaluation

In [ ]:
include("./Algorithms/PPO-RNN.jl")
evaluate(env, agent; num_episodes=10000, max_steps=100) 

## (Todo) Save or plot the data from Train (rewards, losses, evals)